In [1]:

!pip install torchvision 
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets,transforms
from torch.utils.data import DataLoader,random_split
import os
from PIL import Image


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


data_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


data_dir = r"C:\Users\Nisarga\Desktop\project\archive\dataset_blood_group"

image_dataset = datasets.ImageFolder(data_dir, transform=data_transform)


train_size = int(0.8 * len(image_dataset))
val_size = len(image_dataset) - train_size
train_dataset, val_dataset = random_split(image_dataset, [train_size, val_size])


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=8, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=8, pin_memory=True)

dataloaders = {'train': train_loader, 'val': val_loader}


class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.fc1 = nn.Linear(64 * 56 * 56, 512)
        self.fc2 = nn.Linear(512, 8)  # 8 classes for the 8 blood groups

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 56 * 56)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN().to(device)


criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)


num_epochs = 25
for epoch in range(num_epochs):
    print(f'\nEpoch {epoch+1}/{num_epochs}')
    
    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
            print("Training...")
        else:
            model.eval()
            print("Validating...")

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in dataloaders[phase]:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            print(f"Batch done: Loss: {loss.item():.4f}")

        epoch_loss = running_loss / len(dataloaders[phase].dataset)
        epoch_acc = running_corrects.double() / len(dataloaders[phase].dataset)

        print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')


torch.save(model.state_dict(), 'fingerprint_blood_group_model.pth')


^C
Using device: cpu

Epoch 1/25
Training...
Batch done: Loss: 2.0860
Batch done: Loss: 2.1279
Batch done: Loss: 2.1169
Batch done: Loss: 2.0518
Batch done: Loss: 2.1143
Batch done: Loss: 2.0078
Batch done: Loss: 1.9706
Batch done: Loss: 2.1905
Batch done: Loss: 2.1392
Batch done: Loss: 1.9293
Batch done: Loss: 1.9732
Batch done: Loss: 1.9798
Batch done: Loss: 1.9464
Batch done: Loss: 1.9625
Batch done: Loss: 1.8806
Batch done: Loss: 1.8164
Batch done: Loss: 2.0110
Batch done: Loss: 2.0120
Batch done: Loss: 1.9320
Batch done: Loss: 1.8938
Batch done: Loss: 1.8813
Batch done: Loss: 1.8852
Batch done: Loss: 1.7097
Batch done: Loss: 1.7502
Batch done: Loss: 1.6001
Batch done: Loss: 1.8669
Batch done: Loss: 1.6455
Batch done: Loss: 1.6616
Batch done: Loss: 1.6177
Batch done: Loss: 1.6563
Batch done: Loss: 1.4204
Batch done: Loss: 1.3218
Batch done: Loss: 1.5715
Batch done: Loss: 1.6351
Batch done: Loss: 1.6993
Batch done: Loss: 1.6391
Batch done: Loss: 1.7120
Batch done: Loss: 1.4113
Batch

In [2]:
from tkinter import Tk, filedialog, messagebox

In [ ]:
# Prediction function
def predict_image(image_path, model):
    image = Image.open(image_path).convert('RGB')
    image = data_transform(image)
    image = image.unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(image)
        _, preds = torch.max(outputs, 1)
    
    return preds.item()

# Function to upload image and predict blood group
def upload_and_predict():
    root = Tk()
    root.withdraw()  # Hide the root window
    file_path = filedialog.askopenfilename(filetypes=[("Image files", "*.bmp;*.jpg;*.jpeg;*.png")])
    
    if not file_path:
        messagebox.showinfo("Info", "No file selected")
        return
    
    predicted_class = predict_image(file_path, model)
    blood_groups = ['A+', 'A-', 'AB+', 'AB-', 'B+', 'B-', 'O+', 'O-']
    result = f'The predicted blood group is: {blood_groups[predicted_class]}'
    messagebox.showinfo("Prediction Result", result)

# Call the upload and predict function
upload_and_predict()